# 第 6 章互动实验：已知平移的稠密光流

本实验生成两帧固定随机纹理，真实位移已知，并调用 Farneback 作为观察基线。它不提供 `6-1_前向反向筛选.py` 的实现。先拖动位移和窗口看流场；后面三个实验对应教程 6.3 到 6.5 节的实测，其中实验 3 要用到你自己写的 `filter_tracks`。

标题里的 EPE 是去掉边缘后全部像素的平均值。画面里那个圆和矩形的内部是平坦的，稠密光流在那里输出接近 0，平均值主要是被它们拉高的；实验 1 的表格同时给出中位数。

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(2026)
texture = rng.normal(110, 32, (120, 160)).astype(np.float32)
texture = cv2.GaussianBlur(texture, (5, 5), 0.8)
cv2.circle(texture, (55, 58), 20, 220, -1)
cv2.rectangle(texture, (95, 28), (135, 88), 40, -1)
frame0 = np.clip(texture, 0, 255).astype(np.uint8)

In [ ]:
def show_flow(dx=5, dy=3, winsize=15):
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    frame1 = cv2.warpAffine(frame0, M, (frame0.shape[1], frame0.shape[0]), borderMode=cv2.BORDER_REFLECT)
    flow = cv2.calcOpticalFlowFarneback(frame0, frame1, None, 0.5, 4, winsize, 3, 5, 1.2, 0)
    margin = max(abs(dx), abs(dy), 10) + 4
    core = flow[margin:-margin, margin:-margin]
    median = np.median(core.reshape(-1, 2), axis=0)
    endpoint_error = np.linalg.norm(core - np.array([dx, dy], np.float32), axis=2).mean()
    step = 12
    yy, xx = np.mgrid[0:frame0.shape[0]:step, 0:frame0.shape[1]:step]
    sampled = flow[::step, ::step]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(frame0, cmap='gray'); axes[0].set_title('前一帧')
    axes[1].imshow(frame1, cmap='gray'); axes[1].set_title(f'后一帧：真实 ({dx}, {dy})')
    axes[2].imshow(frame1, cmap='gray')
    axes[2].quiver(xx, yy, sampled[...,0], sampled[...,1], color='lime', angles='xy', scale_units='xy', scale=1)
    axes[2].set_title(f'中位数 ({median[0]:.2f}, {median[1]:.2f})，EPE={endpoint_error:.2f}')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()

interact(
    show_flow,
    dx=IntSlider(min=-18, max=18, step=1, value=5),
    dy=IntSlider(min=-12, max=12, step=1, value=3),
    winsize=IntSlider(min=5, max=31, step=2, value=15),
);

## 接上自己的作业

下面的实验有一部分要调用你在 `6-1_前向反向筛选.py` 里写的 `filter_tracks`。先运行这个单元把作业文件加载进来。作业还没写完也可以往下运行，用到它的单元会打印一行提示，不会报错；写完后回来重新运行这个单元。

每个实验都是先写下预测，再运行，然后把结果和原因记到 `实验记录.md`。

In [ ]:
import importlib.util, os
from pathlib import Path

CHAPTER_NAME = "06_光流"
ASSIGNMENT = "6-1_前向反向筛选.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

def load_assignment():
    """加载作业文件；设置了环境变量 COURSE_IMPL_DIR 时改用那里的实现（维护者验证用）。"""
    chapter = find_chapter(CHAPTER_NAME)
    override = os.environ.get("COURSE_IMPL_DIR")
    base = chapter
    if override:
        # 相对路径先按当前目录找，找不到再按课程根目录找
        roots = [Path(override), chapter.parent / override]
        base = next((r / chapter.name for r in roots if (r / chapter.name / ASSIGNMENT).is_file()), roots[0] / chapter.name)
    spec = importlib.util.spec_from_file_location(Path(ASSIGNMENT).stem.replace("-", "_"), base / ASSIGNMENT)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，不中断 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

mod = load_assignment()   # 改了作业文件之后，重新运行这个单元再往下做
print("作业文件：", find_chapter(CHAPTER_NAME) / ASSIGNMENT)


## 实验 1：窗口大小在什么位移下才比得出差别

如果直接把位移调到最大再比较小窗口和大窗口，三种窗口都失败，比不出差别。这个现象本身是教程 6.4、6.5 节的一个结论（窗口不是搜索范围），所以保留在下面表格的最后一行；窗口之间的差别要在中等位移下看。

先预测：

1. 位移 (5, 3) 和 (8, 5) 下，`winsize` 取 5、15、31，EPE 谁大谁小？
2. 位移加到 (10, 6) 和滑块的最大值 (18, 12)，哪一种窗口还能跟上？
3. 最大位移下把 `levels` 从 1 改到 8，结果会变吗？同样的画面放大到 640×480，`levels` 取 1 和 4 又会怎样？

运行后对照教程 6.5 节关于本 Notebook 的那一段。

In [ ]:
def make_frame(scale=1):
    """和最上面相同的画面。scale=4 时画布、圆和矩形都放大 4 倍，纹理的粗细不变。"""
    rng = np.random.default_rng(2026)
    tex = rng.normal(110, 32, (120 * scale, 160 * scale)).astype(np.float32)
    tex = cv2.GaussianBlur(tex, (5, 5), 0.8)
    cv2.circle(tex, (55 * scale, 58 * scale), 20 * scale, 220, -1)
    cv2.rectangle(tex, (95 * scale, 28 * scale), (135 * scale, 88 * scale), 40, -1)
    return np.clip(tex, 0, 255).astype(np.uint8)

def farneback_epe(first, dx, dy, winsize=15, levels=4, change=None):
    """平移 (dx, dy) 得到第二帧（可选地再改亮度），返回 EPE 的平均值和中位数（和互动单元一样去掉边缘）。"""
    second = cv2.warpAffine(first, np.float32([[1, 0, dx], [0, 1, dy]]), (first.shape[1], first.shape[0]), borderMode=cv2.BORDER_REFLECT)
    if change is not None:
        second = change(second)
    flow = cv2.calcOpticalFlowFarneback(first, second, None, 0.5, levels, winsize, 3, 5, 1.2, 0)
    margin = max(abs(dx), abs(dy), 10) + 4
    error = np.linalg.norm(flow[margin:-margin, margin:-margin] - np.array([dx, dy], np.float32), axis=2)
    return float(error.mean()), float(np.median(error))

print('EPE 的平均值，括号里是中位数，单位 px')
for dx, dy in [(5, 3), (8, 5), (10, 6), (18, 12)]:
    cells = []
    for winsize in (5, 15, 31):
        mean, median = farneback_epe(frame0, dx, dy, winsize)
        cells.append(f'winsize {winsize}：{mean:.3f}（{median:.3f}）')
    print(f'位移 ({dx}, {dy})，长度 {np.hypot(dx, dy):.1f} px｜' + '，'.join(cells))

by_levels = [round(farneback_epe(frame0, 18, 12, 15, levels)[0], 3) for levels in (1, 2, 4, 8)]
print('位移 (18, 12)，winsize 15，levels 取 1、2、4、8 的 EPE 平均值：', by_levels)

big = make_frame(4)
for levels in (1, 4):
    mean, median = farneback_epe(big, 18, 12, 15, levels)
    print(f'同样的画面放大到 {big.shape[1]}x{big.shape[0]}，位移 (18, 12)，levels={levels}：平均 {mean:.3f}，中位数 {median:.3f}')

## 实验 2：第二帧整体变亮，谁受影响

如果只用 Farneback，把后一帧整体变亮再算，EPE 没有变化，看不到“亮度恒定假设失效”。Farneback 对整体加一个常数不敏感，这是教程 6.5 节讲的内容，所以下面保留 Farneback 这一栏，再加上稀疏 LK 做对照，现象在 LK 那几栏看。

位移固定为 (5, 3)。LK 跟踪 `goodFeaturesToTrack` 在第一帧取的角点，`maxLevel=3`，看 EPE 的中位数。

先预测：第二帧整体加 10、加 20、乘 1.1 之后，Farneback 的 EPE 变不变？LK 窗口 9、21、31 的误差各往哪边变，哪一种窗口会有大批的点跟丢？窗口从 21 加到 31 还有没有好处？

运行后对照教程 6.5 节的亮度表和它下面关于本 Notebook 的那一段。

In [ ]:
shift = (5, 3)
frame1 = cv2.warpAffine(frame0, np.float32([[1, 0, shift[0]], [0, 1, shift[1]]]), (frame0.shape[1], frame0.shape[0]), borderMode=cv2.BORDER_REFLECT)
corners = cv2.goodFeaturesToTrack(frame0, 200, 0.01, 5)

def lk_epe(second, window):
    """LK 跟踪全部角点，返回 EPE 的中位数和超过 1 px 的点所占的百分比。"""
    moved, _, _ = cv2.calcOpticalFlowPyrLK(frame0, second, corners, None, winSize=(window, window), maxLevel=3)
    error = np.linalg.norm((moved - corners).reshape(-1, 2) - np.float32(shift), axis=1)
    return float(np.median(error)), float(100 * (error > 1).mean())

changes = [('不变', None), ('整体加 10', lambda f: cv2.add(f, 10)), ('整体加 20', lambda f: cv2.add(f, 20)),
           ('整体乘 1.1', lambda f: cv2.convertScaleAbs(f, alpha=1.1))]
print(f'{len(corners)} 个角点。Farneback 一栏是互动单元标题里的 EPE（平均值，px）；LK 三栏是角点上 EPE 的中位数（px），括号里是误差超过 1 px 的点所占的比例')
for name, change in changes:
    second = frame1 if change is None else change(frame1)
    farneback_mean, _ = farneback_epe(frame0, shift[0], shift[1], 15, 4, change)
    cells = []
    for window in (9, 21, 31):
        median, bad = lk_epe(second, window)
        cells.append(f'LK 窗口 {window}：{median:.3f}（{bad:.1f}%）')
    print(f'{name}｜Farneback：{farneback_mean:.3f}｜' + '，'.join(cells))

## 实验 3：往返一致就是跟对了吗（用到作业 6-1）

还是这两帧，位移 (5, 3)。对每个角点前向跟一次、再反向跟回来，用你的 `filter_tracks` 按阈值 0.1、1、5 px 筛选。真值已知，所以能数出留下来的点里有几个其实是错的（EPE 不小于 1 px）。

做两遍：一遍 `maxLevel=3`，一遍 `maxLevel=0`（不用金字塔，这张图的纹理很细，5.8 px 的位移已经超出单层 LK 的范围）。

先预测：

1. `maxLevel=0` 时 `status` 还是 1 的有多少，真正跟对的有多少？
2. 阈值 1 px 留下的点里有没有错的？阈值收紧到 0.1 px 呢？代价是什么？
3. 圆和矩形内部有没有角点？

运行后对照教程 6.3 节“前向-反向误差的阈值怎么定”。

In [ ]:
def round_trip(max_level):
    forward, ok1, _ = cv2.calcOpticalFlowPyrLK(frame0, frame1, corners, None, winSize=(21, 21), maxLevel=max_level)
    backward, ok2, _ = cv2.calcOpticalFlowPyrLK(frame1, frame0, forward, None, winSize=(21, 21), maxLevel=max_level)
    return forward, backward, (ok1.ravel() != 0) & (ok2.ravel() != 0)

for max_level in (3, 0):
    forward, backward, both_ok = round_trip(max_level)
    error = np.linalg.norm((forward - corners).reshape(-1, 2) - np.float32(shift), axis=1)
    print(f'maxLevel={max_level}：{len(corners)} 个角点里 status 为 1 的 {int(both_ok.sum())} 个，真正跟对（EPE 小于 1 px）的 {int((error < 1).sum())} 个')
    for threshold in (0.1, 1.0, 5.0):
        keep = attempt(mod.filter_tracks, corners, forward, backward, both_ok, threshold)
        if keep is None:
            break
        print(f'    阈值 {threshold} px：留下 {int(keep.sum())} 个，其中跟错的 {int((keep & (error >= 1)).sum())} 个')

xy = np.rint(corners.reshape(-1, 2)).astype(int)
inside = ((xy[:, 0] - 55) ** 2 + (xy[:, 1] - 58) ** 2 < 14 ** 2) | ((xy[:, 0] > 101) & (xy[:, 0] < 129) & (xy[:, 1] > 34) & (xy[:, 1] < 82))
print(f'落在圆和矩形平坦内部的角点：{int(inside.sum())} 个')

## 破坏实验与记录

1. 实验 1：窗口之间的差别在位移 (5, 3) 和 (8, 5) 下看；位移到 (10, 6) 以上三种窗口都失败。把互动单元的滑块拉到这几个位置各看一眼流场，再解释为什么 160×120 的图上改 `levels` 没有用。
2. 实验 2：把 `changes` 里的“整体加 20”改成“整体加 40”，这时灰度 220 的圆会饱和到 255。先预测 Farneback 和 LK 各会怎样，再运行。
3. 实验 3：把位移改成 (10, 6)，`maxLevel` 取 0、1、3，看阈值 1 px 留下的错点数怎么变。
4. 比较边界与内部区域误差，解释为什么评估时排除了边缘。

把真实位移、估计中位数、EPE 和失败原因写入 `实验记录.md`。